In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/movies_metadata.csv')

# Print all the features/columns of the DataFrame

df.columns

/var/folders/y8/ywn__z7s44n4f3jpljgjl8rh0000gn/T/ipykernel_36443/1738917086.py:4: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/movies_metadata.csv')


Index(['adult', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id',
       'imdb_id', 'original_language', 'original_title', 'overview',
       'popularity', 'poster_path', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'video',
       'vote_average', 'vote_count'],
      dtype='object')

In [2]:
# Only keep features we require

df = df[['title', 'genres', 'release_date',
         'runtime', 'vote_average', 'vote_count']]

df.head()

,title,genres,release_date,runtime,vote_average,vote_count
0,Toy Story,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",1995-10-30,81.0,7.7,5415.0
1,Jumanji,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",1995-12-15,104.0,6.9,2413.0
2,Grumpier Old Men,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",1995-12-22,101.0,6.5,92.0
3,Waiting to Exhale,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",1995-12-22,127.0,6.1,34.0
4,Father of the Bride Part II,"[{'id': 35, 'name': 'Comedy'}]",1995-02-10,106.0,5.7,173.0


In [3]:
# Convert release_date into pandas datetime formate
df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')

# Extract year from the datetime
df['year'] = df['release_date'].dt.year.astype('Int64')

df['year']

0        1995
1        1995
2        1995
3        1995
4        1995
         ... 
45461    <NA>
45462    2011
45463    2003
45464    1917
45465    2017
Name: year, Length: 45466, dtype: Int64

In [4]:
# Convert NaT to 0
df['year'] = df['year'].fillna(0)

In [5]:
# Drop the release_date column
df = df.drop('release_date', axis=1)

# Display the dataframe
df.head()

,title,genres,runtime,vote_average,vote_count,year
0,Toy Story,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",81.0,7.7,5415.0,1995
1,Jumanji,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",104.0,6.9,2413.0,1995
2,Grumpier Old Men,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",101.0,6.5,92.0,1995
3,Waiting to Exhale,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",127.0,6.1,34.0,1995
4,Father of the Bride Part II,"[{'id': 35, 'name': 'Comedy'}]",106.0,5.7,173.0,1995


In [6]:
# Print genres of the first movie
df.iloc[0]['genres']

"[{'id': 16, 'name': 'Animation'}, {'id': 35, 'name': 'Comedy'}, {'id': 10751, 'name': 'Family'}]"

In [7]:
# trying to learn about literal_eval function from ast library

from ast import literal_eval

# Define a stringified list and output its type
a = "[1,2,3]"
print(type(a))

# Apply literal_eval and output type
b = literal_eval(a)
print(type(b))

<class 'str'>
<class 'list'>


In [8]:
# Convert all NaN into stringified empty lists
df['genres'] = df['genres'].fillna('[]')

# Apply literal_eval to convert to the list object
df['genres'] = df['genres'].apply(literal_eval)

# Convert list of dictionaries to a list of strings
df['genres'] = df['genres'].apply(
    lambda x: [i['name'].lower() for i in x] if isinstance(x, list) else [])

In [9]:
df.head()

,title,genres,runtime,vote_average,vote_count,year
0,Toy Story,"[animation, comedy, family]",81.0,7.7,5415.0,1995
1,Jumanji,"[adventure, fantasy, family]",104.0,6.9,2413.0,1995
2,Grumpier Old Men,"[romance, comedy]",101.0,6.5,92.0,1995
3,Waiting to Exhale,"[comedy, drama, romance]",127.0,6.1,34.0,1995
4,Father of the Bride Part II,[comedy],106.0,5.7,173.0,1995


In [11]:
# Convert the cleaned (non-exploded) dataframe df into a CSV file and save it in the data folder
# Set parameter index to False as the index of the DataFrame has no inherent meaning.
df.to_csv('../data/metadata_cleaned.csv', index=False)

In [15]:
# Create a new feature by exploding genres
exp_df = df.explode('genres')

# rename the column
exp_df = exp_df.rename(columns={'genres': 'genre'})

exp_df.head()

,title,genre,runtime,vote_average,vote_count,year
0,Toy Story,animation,81.0,7.7,5415.0,1995
0,Toy Story,comedy,81.0,7.7,5415.0,1995
0,Toy Story,family,81.0,7.7,5415.0,1995
1,Jumanji,adventure,104.0,6.9,2413.0,1995
1,Jumanji,fantasy,104.0,6.9,2413.0,1995


In [16]:
def build_chart(exp_df, percentile=0.8):
    # Ask for preferred genres
    print("Input preferred genre")
    genre = input()

    # Ask for lower limit of duration
    print("Input shortest duration")
    low_time = int(input())

    # Ask for upper limit of duration
    print("Input longest duration")
    high_time = int(input())

    # Ask for lower limit of timeline
    print("Input earliest year")
    low_year = int(input())

    # Ask for upper limit of timeline
    print("Input latest year")
    high_year = int(input())

    # define a new movies variable to store the preferred movies
    movies = exp_df.copy()

    # Filter based on the condition
    movies = movies[(movies['genre'] == genre) &
                    (movies['runtime'] >= low_time) &
                    (movies['runtime'] <= high_time) &
                    (movies['year'] >= low_year) &
                    (movies['year'] <= high_year)]

    # Compute the values of C and m for the filtered movies
    C = movies['vote_average'].mean()
    m = movies['vote_count'].quantile(percentile)

    # Only consider movies that have higher than m votes.
    q_movies = movies.copy().loc[movies['vote_count'] >= m]

    # Calculate score using the IMDB formula
    q_movies['score'] = (q_movies['vote_count']/(q_movies['vote_count']+m)
                         * q_movies['vote_average'] + m/(q_movies['vote_count'] + m)*C)

    # Sort the movies in descending order of their scores
    q_movies = q_movies.sort_values('score', ascending=False)

    return q_movies

In [17]:
# Generate the chart for the top action movies and display top 5.
build_chart(exp_df).head()

Input preferred genre
Input shortest duration
Input longest duration
Input earliest year
Input latest year


,title,genre,runtime,vote_average,vote_count,year,score
4863,The Lord of the Rings: The Fellowship of the Ring,action,178.0,8.0,8892.0,2001,7.946706
5814,The Lord of the Rings: The Two Towers,action,179.0,8.0,7641.0,2002,7.938201
2458,The Matrix,action,136.0,7.9,9079.0,1999,7.849915
3456,Gladiator,action,155.0,7.9,5566.0,2000,7.819389
9430,Oldboy,action,120.0,8.0,2000.0,2003,7.779599


In [18]:
# for animated movies between 30 minutes and 2 hours in length and released anywhere between 1990 and 2005
build_chart(exp_df).head()

Input preferred genre
Input shortest duration
Input longest duration
Input earliest year
Input latest year


,title,genre,runtime,vote_average,vote_count,year,score
5481,Spirited Away,animation,125.0,8.3,3968.0,2001,8.181993
9698,Howl's Moving Castle,animation,119.0,8.2,2049.0,2004,7.992967
2884,Princess Mononoke,animation,134.0,8.2,2041.0,1997,7.992239
359,The Lion King,animation,89.0,8.0,5520.0,1994,7.925978
13724,Up,animation,96.0,7.8,7048.0,2009,7.747913
